# OpenOrca Query Intelligence — Exploratory Data Analysis

This notebook walks through the dataset, visualises class distributions,
text length distributions, and surfaces patterns that informed the feature
engineering decisions in `src/features/engineer.py`.

**Run `python train.py` (or `make data`) before executing this notebook.**

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.data.loader import load_raw
from src.data.preprocessor import clean_text, preprocess

pd.set_option('display.max_colwidth', 120)
print('Imports OK ✓')

## 1. Data Loading

In [ ]:
df = load_raw()
print(f'Shape: {df.shape}')
df.head(3)

In [ ]:
df.dtypes

In [ ]:
print('Missing values:')
df.isnull().sum()

## 2. Class Distribution

In [ ]:
src_counts = df['source_label'].value_counts()
fig = px.bar(
    x=src_counts.index, y=src_counts.values,
    title='Query Source Distribution',
    labels={'x': 'Source', 'y': 'Count'},
    color=src_counts.values,
    color_continuous_scale='teal',
    template='plotly_white',
)
fig.update_layout(coloraxis_showscale=False)
fig.show()

## 3. Text Length Analysis

In [ ]:
df['q_len']    = df['question'].str.len()
df['q_words']  = df['question'].str.split().str.len()
df['r_words']  = df['response'].str.split().str.len()

fig = make_subplots(rows=1, cols=2, subplot_titles=(
    'Q char length by source', 'Q word count by source'
))
for i, (col, name) in enumerate([('q_len','Char length'), ('q_words','Word count')], 1):
    for src in df['source_label'].unique():
        fig.add_trace(
            go.Box(y=df[df['source_label']==src][col], name=src, showlegend=(i==1)),
            row=1, col=i
        )
fig.update_layout(height=450, template='plotly_white', title='Question Lengths by Source')
fig.show()

## 4. Response Complexity Distribution

In [ ]:
if 'complexity_label' in df.columns:
    ct = pd.crosstab(df['source_label'], df['complexity_label'], normalize='index') * 100
    fig = px.imshow(
        ct.round(1), text_auto=True,
        title='Response Complexity (%) by Source',
        color_continuous_scale='Blues',
        template='plotly_white',
    )
    fig.show()

## 5. Top Unigrams per Class

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import re

def top_terms(corpus, n=20):
    vec = TfidfVectorizer(max_features=5000, stop_words='english', ngram_range=(1,1))
    X = vec.fit_transform(corpus)
    scores = X.mean(axis=0).A1
    terms  = vec.get_feature_names_out()
    idx    = scores.argsort()[::-1][:n]
    return pd.DataFrame({'term': terms[idx], 'score': scores[idx]})

for src in df['source_label'].unique():
    corpus = df[df['source_label']==src]['question'].tolist()
    top = top_terms(corpus)
    fig = px.bar(
        top, x='score', y='term', orientation='h',
        title=f'Top Terms — {src}',
        template='plotly_white',
        color='score', color_continuous_scale='teal',
    )
    fig.update_layout(height=420, coloraxis_showscale=False)
    fig.show()

## 6. Preprocessing Check

In [ ]:
df_proc, le = preprocess(df.copy(), target='source_label')
print('Classes:', list(le.classes_))
print('Shape after preprocessing:', df_proc.shape)
df_proc[['question', 'question_clean', 'label_name', 'target']].head(5)